# EBITDA Normalization with Adjustments

This notebook mirrors `examples/scripts/statements/adjustments_demo.py` and covers:

1. Building a simple private-credit statements model.
2. Defining fixed, capped, and percentage-based adjustments.
3. Normalizing EBITDA and merging the adjusted metric back into the results.



## Setup

Import finstack along with the statements adjustment helpers and the `PeriodId` utility for quarter labels.



In [ ]:
import finstack
from finstack.statements import ModelBuilder, Evaluator
from finstack.statements.adjustments import (
    Adjustment,
    NormalizationConfig,
    NormalizationEngine,
)
from finstack import dates

PeriodId = dates.PeriodId



## Build the base statements model

Create quarterly periods for 2025, hard-code Revenue, and derive EBITDA via a compute expression.



In [ ]:
builder = ModelBuilder.new("private_credit_deal")
builder.periods("2025Q1..Q4", None)

revenue_schedule = {
    PeriodId.quarter(2025, 1): 1000.0,
    PeriodId.quarter(2025, 2): 1100.0,
    PeriodId.quarter(2025, 3): 1200.0,
    PeriodId.quarter(2025, 4): 1300.0,
}

builder.value_scalar("Revenue", revenue_schedule)
builder.compute("EBITDA", "Revenue * 0.2")

model = builder.build()
model



## Evaluate the model

Run the evaluator to produce base metrics for each quarter.



In [ ]:
evaluator = Evaluator.new()
results = evaluator.evaluate(model)

periods = [PeriodId.quarter(2025, q) for q in range(1, 5)]
base_view = [
    {
        "Period": str(period),
        "Revenue": results.get("Revenue", period),
        "EBITDA": results.get("EBITDA", period),
    }
    for period in periods
]
base_view



## Define adjustments

The demo uses three common adjustments:

- Owner's compensation (fixed add-back).
- Synergies (fixed add-back capped at 20% of EBITDA).
- Legal fees (1% of Revenue each quarter).



In [ ]:
owners_comp = Adjustment.fixed(
    "owners_comp",
    "Owner's Compensation",
    {
        "2025Q1": 50.0,
        "2025Q2": 50.0,
        "2025Q3": 50.0,
        "2025Q4": 50.0,
    },
)

synergies = Adjustment.fixed(
    "synergies",
    "Synergies",
    {
        "2025Q1": 100.0,
        "2025Q2": 100.0,
        "2025Q3": 100.0,
        "2025Q4": 100.0,
    },
).with_cap("EBITDA", 0.20)

legal_fees = Adjustment.percentage(
    "legal",
    "Legal Fees",
    "Revenue",
    0.01,
)

owners_comp, synergies, legal_fees



## Configure and run normalization

Attach the adjustments to a `NormalizationConfig`, run the engine, and capture the audit trail for each quarter.



In [ ]:
config = NormalizationConfig("EBITDA")
config.add_adjustment(owners_comp)
config.add_adjustment(synergies)
config.add_adjustment(legal_fees)

normalization_results = NormalizationEngine.normalize(results, config)


def format_adjustments(result):
    lines = []
    for adj in result.adjustments:
        cap_note = f" (capped from {adj.raw_amount:.2f})" if adj.is_capped else ""
        lines.append(f"+ {adj.name}: {adj.capped_amount:.2f}{cap_note}")
    return "\n".join(lines)


audit_rows = [
    {
        "Period": str(res.period),
        "Base EBITDA": float(res.base_value),
        "Adjusted EBITDA": float(res.final_value),
        "Adjustments": format_adjustments(res),
    }
    for res in normalization_results
]

audit_rows



## Merge adjusted EBITDA back

`NormalizationEngine.merge_into_results` stamps the adjusted series into the model outputs so downstream metrics can reference it.



In [ ]:
NormalizationEngine.merge_into_results(
    results,
    normalization_results,
    "Adjusted EBITDA",
)

adjusted_series = [
    {
        "Period": str(period),
        "Adjusted EBITDA": results.get("Adjusted EBITDA", period),
    }
    for period in periods
]
adjusted_series

